# 02. 전처리

`01_data_merge.ipynb`에서 만든 `merged_data.csv`를 분석 가능한 형태로 정리한다.
- 결측치, 중복, 병합 누락 여부를 다시 확인한다.
- 범죄율과 인구 대비 비율 변수를 만든다.
- 리뷰용 최종 데이터와 회귀분석용 표준화 데이터를 따로 저장한다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# 노트북 실행 위치가 달라도 같은 파일을 읽도록 프로젝트 루트를 고정한다.
PROJECT_ROOT = Path('/Users/ijaejun/Documents/sophomore_high/crime_catchers')
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MERGED_PATH = DATA_DIR / 'merged_data.csv'
FINAL_PATH = DATA_DIR / 'final_data.csv'
MODEL_PATH = DATA_DIR / 'model_data_scaled.csv'

print('프로젝트 위치:', PROJECT_ROOT)
print('입력 파일:', MERGED_PATH)


프로젝트 위치: /Users/ijaejun/Documents/sophomore_high/crime_catchers
입력 파일: /Users/ijaejun/Documents/sophomore_high/crime_catchers/data/processed/merged_data.csv


## 1. 데이터 로드 및 기본 구조 확인

In [2]:
df = pd.read_csv(MERGED_PATH, encoding='utf-8-sig')

CRIME_COLUMNS = ['살인기수', '강도', '강간', '절도', '폭행']

# 병합 노트북을 다시 실행하기 전 파일에는 개별 범죄 컬럼만 있을 수 있다.
# 이 경우 전처리에서 5대범죄합계를 만들어 뒤 단계가 끊기지 않게 한다.
if '5대범죄합계' not in df.columns and set(CRIME_COLUMNS).issubset(df.columns):
    df[CRIME_COLUMNS] = df[CRIME_COLUMNS].apply(pd.to_numeric, errors='coerce')
    df['5대범죄합계'] = df[CRIME_COLUMNS].sum(axis=1).astype(int)

EXPECTED_COLUMNS = [
    '지역', '연도', '5대범죄합계',
    '실업률', '음주율', '물가상승률', '인구수', '평균기온', '경찰1인당주민수',
    '기초수급자수', '조이혼율', '지역소득', '외국인수', '외국인비율(%)'
]

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
extra_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))

print('shape:', df.shape)
print('누락 컬럼:', missing_columns)
print('추가 컬럼:', extra_columns)
print(df.head())

if missing_columns:
    raise ValueError(f'필수 컬럼이 없습니다: {missing_columns}')


shape: (66, 14)
누락 컬럼: []
추가 컬럼: []
   지역    연도  5대범죄합계  실업률   음주율  물가상승률      인구수  평균기온  경찰1인당주민수  기초수급자수  조이혼율  \
0  광주  2014   16364  2.8  60.6    1.6  1475884  14.3     476.7   59598   2.1   
1  광주  2015   14095  2.9  61.9    0.3  1472199  14.6     458.6   71683   1.9   
2  광주  2016   11073  3.1  58.6    0.9  1469214  15.0     454.6   69420   1.9   
3  광주  2017    9916  2.9  61.6    2.1  1463770  14.6     447.5   65712   1.8   
4  광주  2018   10070  3.8  60.3    1.2  1459336  14.6     439.6   72757   2.0   

    지역소득   외국인수  외국인비율(%)  
0  23448  17064    1.1562  
1  24731  18455    1.2536  
2  26248  19920    1.3558  
3  27449  21279    1.4537  
4  28594  22815    1.5634  


## 2. 품질 점검

`inner merge` 결과만 보면 누락 행이 숨어 있을 수 있으므로, 전처리 시작 전에 현재 파일 기준의 행 수와 결측/중복을 명확히 확인한다.


In [3]:
# 6개 광역시 x 2014~2024년 = 66행이어야 한다.
expected_rows = 6 * 11
key_duplicates = df.duplicated(subset=['지역', '연도']).sum()
missing_by_column = df.isna().sum()

print('예상 행 수:', expected_rows)
print('실제 행 수:', len(df))
print('지역-연도 중복 수:', key_duplicates)
print()
print('결측치 개수:')
print(missing_by_column)

if len(df) != expected_rows:
    raise ValueError(f'행 수가 예상과 다릅니다. expected={expected_rows}, actual={len(df)}')
if key_duplicates > 0:
    raise ValueError('지역-연도 기준 중복 행이 있습니다.')
if missing_by_column.sum() > 0:
    raise ValueError('결측치가 있습니다. 먼저 원본/병합 과정을 확인하세요.')


예상 행 수: 66
실제 행 수: 66
지역-연도 중복 수: 0

결측치 개수:
지역          0
연도          0
5대범죄합계      0
실업률         0
음주율         0
물가상승률       0
인구수         0
평균기온        0
경찰1인당주민수    0
기초수급자수      0
조이혼율        0
지역소득        0
외국인수        0
외국인비율(%)    0
dtype: int64


In [4]:
# 문자열로 들어온 '-' 또는 빈칸은 isna()에서 안 잡힐 수 있어서 별도로 확인한다.
MISSING_MARKERS = {'', '-', 'NA', 'N/A', 'null', 'NULL', 'None', 'nan'}
marker_counts = {}

for col in df.columns:
    values = df[col].astype(str).str.strip()
    count = values.isin(MISSING_MARKERS).sum()
    if count > 0:
        marker_counts[col] = int(count)

numeric_columns = [col for col in EXPECTED_COLUMNS if col not in ['지역']]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')
inf_count = np.isinf(df[numeric_columns].to_numpy()).sum()

print('숨은 결측 표시:', marker_counts)
print('숫자 변환 후 결측치 합계:', int(df[numeric_columns].isna().sum().sum()))
print('inf 개수:', int(inf_count))

if marker_counts:
    raise ValueError(f'문자열 결측 표시가 있습니다: {marker_counts}')
if df[numeric_columns].isna().sum().sum() > 0:
    raise ValueError('숫자 변환 과정에서 결측치가 생겼습니다.')
if inf_count > 0:
    raise ValueError('무한대 값이 있습니다. 인구수 0 여부 등을 확인하세요.')


숨은 결측 표시: {}
숫자 변환 후 결측치 합계: 0
inf 개수: 0


## 3. 분석용 변수 생성

README 기준으로 종속변수는 인구 10만 명당 범죄율로 통일한다. 외국인수와 기초수급자수는 도시 인구 규모의 영향을 줄이기 위해 비율 변수로 바꾼다.


In [5]:
final_df = df.copy()

# 종속변수(Y): 도시 규모 차이를 줄이기 위해 인구 10만 명당 범죄 발생 건수로 변환한다.
final_df['범죄율'] = (final_df['5대범죄합계'] / final_df['인구수'] * 100000).round(2)

# 독립변수(X): 기초수급자수는 절대값 대신 인구 대비 비율을 사용한다.
final_df['기초수급비율(%)'] = (final_df['기초수급자수'] / final_df['인구수'] * 100).round(4)

ANALYSIS_COLUMNS = [
    '지역', '연도', '범죄율',
    '실업률', '음주율', '물가상승률', '인구수', '평균기온', '경찰1인당주민수',
    '기초수급비율(%)', '조이혼율', '지역소득', '외국인비율(%)'
]

final_df = final_df[ANALYSIS_COLUMNS].sort_values(['지역', '연도']).reset_index(drop=True)

print('분석용 데이터 shape:', final_df.shape)
print(final_df.head(10))


분석용 데이터 shape: (66, 13)
   지역    연도      범죄율  실업률   음주율  물가상승률      인구수  평균기온  경찰1인당주민수  기초수급비율(%)  \
0  광주  2014  1108.76  2.8  60.6    1.6  1475884  14.3     476.7     4.0381   
1  광주  2015   957.41  2.9  61.9    0.3  1472199  14.6     458.6     4.8691   
2  광주  2016   753.67  3.1  58.6    0.9  1469214  15.0     454.6     4.7250   
3  광주  2017   677.43  2.9  61.6    2.1  1463770  14.6     447.5     4.4892   
4  광주  2018   690.04  3.8  60.3    1.2  1459336  14.6     439.6     4.9856   
5  광주  2019   710.76  3.7  61.1    0.2  1456468  14.7     423.8     5.2314   
6  광주  2020   646.87  3.9  52.9    0.4  1450062  14.5     419.0     5.8454   
7  광주  2021   599.47  3.6  54.5    2.6  1441611  15.1     404.0     6.3504   
8  광주  2022   614.72  2.9  58.6    5.1  1431050  14.8     401.0     6.5782   
9  광주  2023   684.24  2.5  59.5    3.7  1419237  15.3     398.9     6.7975   

   조이혼율   지역소득  외국인비율(%)  
0   2.1  23448    1.1562  
1   1.9  24731    1.2536  
2   1.9  26248    1.3558  
3   1.8  

## 4. 최종 검증 및 저장

In [ ]:
final_missing = final_df.isna().sum()
final_inf = np.isinf(final_df.select_dtypes(include='number').to_numpy()).sum()

print('최종 결측치:')
print(final_missing)
print('최종 inf 개수:', int(final_inf))

if final_missing.sum() > 0:
    raise ValueError('최종 데이터에 결측치가 남아 있습니다.')
if final_inf > 0:
    raise ValueError('최종 데이터에 inf 값이 남아 있습니다.')

DATA_DIR.mkdir(parents=True, exist_ok=True)
final_df.to_csv(FINAL_PATH, index=False, encoding='utf-8-sig')
print('저장 완료:', FINAL_PATH)


## 5. 회귀분석용 데이터 만들기

회귀분석에서는 변수 단위가 다르면 계수 비교가 어려우므로 독립변수만 표준화한다. `지역`은 도시 고유 차이를 반영하기 위해 더미변수로 바꾼다.


In [ ]:
FEATURE_COLUMNS = [
    '실업률', '음주율', '물가상승률', '인구수', '평균기온', '경찰1인당주민수',
    '기초수급비율(%)', '조이혼율', '지역소득', '외국인비율(%)'
]

model_df = pd.get_dummies(
    final_df,
    columns=['지역'],
    prefix='지역',
    drop_first=True,
    dtype=int
)

# sklearn이 없어도 실행되도록 pandas로 z-score 표준화를 직접 계산한다.
feature_mean = model_df[FEATURE_COLUMNS].mean()
feature_std = model_df[FEATURE_COLUMNS].std(ddof=0).replace(0, 1)
model_df[FEATURE_COLUMNS] = ((model_df[FEATURE_COLUMNS] - feature_mean) / feature_std).round(6)

model_df.to_csv(MODEL_PATH, index=False, encoding='utf-8-sig')

print('모델용 데이터 shape:', model_df.shape)
print('저장 완료:', MODEL_PATH)
print(model_df.head())


## 6. 코드 리뷰용 체크포인트

- `final_data.csv`: 사람이 해석하기 쉬운 전처리 완료 데이터
- `model_data_scaled.csv`: 회귀분석에 바로 넣기 좋은 표준화 + 지역 더미 데이터
- 2020~2021년은 README 방침대로 제거하지 않고 포함한다.
